<a href="https://colab.research.google.com/github/fhmahi957/Licence-plate-NightVision/blob/main/Licence_plate_NightVision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
import os
import torch

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Create a folder for your models
save_dir = '/content/drive/MyDrive/CSE414_Project2/models'
os.makedirs(save_dir, exist_ok=True)

# 3. Save the trained model weights to Drive
model_path = os.path.join(save_dir, 'faster_rcnn_baseline.pth')
torch.save(model.state_dict(), model_path)

print(f"SUCCESS! Model saved to Google Drive at: {model_path}")

Mounted at /content/drive
SUCCESS! Model saved to Google Drive at: /content/drive/MyDrive/CSE414_Project2/models/faster_rcnn_baseline.pth


In [8]:
# Load cell (run this first)

# 1. Mount Drive and Import Libraries
from google.colab import drive
import torch
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

drive.mount('/content/drive')

# 2. Recreate the Model Architecture (The "Empty Kitchen")
# We must define the exact same architecture we trained
model = fasterrcnn_mobilenet_v3_large_320_fpn(weights=None)
in_features = model.roi_heads.box_predictor.cls_score.in_features
# NUM_CLASSES was 4 (Background + 3 objects from your dataset)
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 4)

# 3. Load the Weights from Drive (Bringing the "Cake" out of the fridge)
model_path = '/content/drive/MyDrive/CSE414_Project2/models/faster_rcnn_baseline.pth'
model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

# 4. Set to Evaluation Mode
model.eval()
print("SUCCESS! Model loaded from Google Drive and ready for testing.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 59.4MB/s]


SUCCESS! Model loaded from Google Drive and ready for testing.


In [13]:
# Evaluation code (run this 2nd)

import os
import torch
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import torchvision.transforms as T
from torchmetrics.detection import MeanAveragePrecision
from tqdm import tqdm

# Paths
NIGHT_IMG_DIR = '/content/nighttime_images'
LBL_DIR = '/content/source_labels' # We use the original labels because geometry didn't change

# 1. Define the Nighttime Dataset
# We iterate through the ORIGINAL filenames, but load the NIGHTTIME images
class NighttimeDataset(Dataset):
    def __init__(self, night_img_dir, lbl_dir):
        self.night_img_dir = night_img_dir
        self.lbl_dir = lbl_dir
        # Get list of original filenames (e.g., 'car1.jpg')
        # We assume the night images are named 'night_car1.jpg'
        self.original_images = sorted([f for f in os.listdir(lbl_dir) if f.endswith('.txt')])

    def __len__(self):
        return len(self.original_images)

    def __getitem__(self, idx):
        orig_name = self.original_images[idx] # e.g., 'car1.txt'
        img_name = orig_name.replace('.txt', '.jpg') # e.g., 'car1.jpg'
        night_img_name = f"night_{img_name}" # e.g., 'night_car1.jpg'

        img_path = os.path.join(self.night_img_dir, night_img_name)
        lbl_path = os.path.join(self.lbl_dir, orig_name)

        image = Image.open(img_path).convert("RGB")
        width, height = image.size

        boxes = []
        labels = []

        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        x_center = float(parts[1])
                        y_center = float(parts[2])
                        w = float(parts[3])
                        h = float(parts[4])

                        x1 = (x_center - w / 2) * width
                        y1 = (y_center - h / 2) * height
                        x2 = (x_center + w / 2) * width
                        y2 = (y_center + h / 2) * height

                        # Clamp
                        x1 = max(0, x1); y1 = max(0, y1)
                        x2 = min(width, x2); y2 = min(height, y2)

                        if x2 > x1 and y2 > y1:
                            boxes.append([x1, y1, x2, y2])
                            labels.append(class_id + 1) # 1-indexed

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx]),
            'area': (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
            'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64)
        }

        # Transform to tensor (same as training)
        transform = T.Compose([T.ToTensor()])
        image = transform(image)

        return image, target

# 2. Setup Data Loader
dataset = NighttimeDataset(NIGHT_IMG_DIR, LBL_DIR)
# Batch size 1 is best for evaluation metrics
data_loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=lambda x: tuple(zip(*x)))

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval() # Crucial for evaluation

# 3. Initialize Metric
metric = MeanAveragePrecision(iou_type="bbox", box_format="xyxy")

print("Starting Domain Shift Analysis (Testing Clean Model on Nighttime Data)...")
print("This measures the 'Before' performance.\n")

# 4. Evaluation Loop
with torch.no_grad():
    for images, targets in tqdm(data_loader, desc="Evaluating"):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Get predictions
        predictions = model(images)

        # Format for torchmetrics
        # predictions is a list of dicts: [{'boxes':..., 'labels':..., 'scores':...}]
        preds_for_metric = [{
            'boxes': p['boxes'].cpu(),
            'labels': p['labels'].cpu(),
            'scores': p['scores'].cpu()
        } for p in predictions]

        targets_for_metric = [{
            'boxes': t['boxes'].cpu(),
            'labels': t['labels'].cpu()
        } for t in targets]

        # Update metric
        metric.update(preds_for_metric, targets_for_metric)

# 5. Compute and Print Results
result = metric.compute()
print("\n DOMAIN SHIFT ANALYSIS RESULTS")
print(f"mAP (mean Average Precision): {result['map'].item():.4f}")
print(f"mAP @ 0.5 IoU: {result['map_50'].item():.4f}")
print(f"mAP @ 0.75 IoU: {result['map_75'].item():.4f}")

print("Note: A low mAP here proves that Domain Shift is a problem!")


Starting Domain Shift Analysis (Testing Clean Model on Nighttime Data)...
This measures the 'Before' performance.



Evaluating: 100%|██████████| 746/746 [00:19<00:00, 38.55it/s]



 DOMAIN SHIFT ANALYSIS RESULTS
mAP (mean Average Precision): 0.1719
mAP @ 0.5 IoU: 0.3133
mAP @ 0.75 IoU: 0.1751
Note: A low mAP here proves that Domain Shift is a problem!


In [10]:
# Installing necessary libraries
!pip install ultralytics albumentations opencv-python-headless matplotlib seaborn tqdm pyyaml torchmetrics -q

In [2]:
import os
import cv2
import torch
import albumentations as A
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Check if the GPU is connected and ready
print("Environment setup complete!")
print(f"PyTorch version: {torch.__version__}")
print(f"Is GPU available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not available. Please go to Runtime > Change runtime type and select T4 GPU.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Environment setup complete!
PyTorch version: 2.11.0+cu130
Is GPU available: True
GPU Name: Tesla T4


In [3]:
from google.colab import files
import zipfile
import os
import shutil

# 1. Upload the ZIP file from your computer
print("Please select the Roboflow ZIP file from Downloads folder...")
uploaded = files.upload()

# Get the name of the uploaded file
zip_filename = list(uploaded.keys())[0]
print(f"Uploaded: {zip_filename}")

# 2. Create clean folders for our project
os.makedirs('/content/source_images', exist_ok=True)
os.makedirs('/content/source_labels', exist_ok=True)

# 3. Unzip and organize the files
print("\nUnzipping and organizing dataset...")
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset_raw')

base_path = '/content/dataset_raw'

for root, dirs, files in os.walk(base_path):
    # If we find the images folder inside a 'train' folder
    if 'images' in dirs and 'train' in root:
        img_dir = os.path.join(root, 'images')
        for img in os.listdir(img_dir):
            if img.lower().endswith(('.jpg', '.jpeg', '.png')):
                shutil.move(os.path.join(img_dir, img), '/content/source_images/')

    # If we find the labels folder inside a 'train' folder
    if 'labels' in dirs and 'train' in root:
        lbl_dir = os.path.join(root, 'labels')
        for lbl in os.listdir(lbl_dir):
            if lbl.lower().endswith('.txt'):
                shutil.move(os.path.join(lbl_dir, lbl), '/content/source_labels/')

# 4. Verify the count
img_count = len(os.listdir('/content/source_images'))
lbl_count = len(os.listdir('/content/source_labels'))

print(f"\nSUCCESS! Dataset organized.")
print(f"Total Images: {img_count}")
print(f"Total Labels: {lbl_count}")

Please select the Roboflow ZIP file from Downloads folder...


Saving bangla-license-plate-detection.v1i.yolov8.zip to bangla-license-plate-detection.v1i.yolov8.zip
Uploaded: bangla-license-plate-detection.v1i.yolov8.zip

Unzipping and organizing dataset...

SUCCESS! Dataset organized.
Total Images: 746
Total Labels: 746


In [4]:
# creating distorted image from clean image
import os
import cv2
import albumentations as A
from tqdm import tqdm

# Define paths
CLEAN_IMG_DIR = '/content/source_images'
NIGHT_IMG_DIR = '/content/nighttime_images'

# Create the nighttime directory if it doesn't exist
os.makedirs(NIGHT_IMG_DIR, exist_ok=True)

# Define the nighttime degradation transform
nighttime_transform = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=(-0.6, -0.2), contrast_limit=(-0.4, 0.0), p=0.9),
    A.GaussianBlur(blur_limit=(3, 7), p=0.6),
    A.MotionBlur(blur_limit=(5, 11), p=0.5),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05, p=0.5)
])

print("Generating synthetic nighttime dataset...")
image_files = [f for f in os.listdir(CLEAN_IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for filename in tqdm(image_files, desc="Processing images"):
    img_path = os.path.join(CLEAN_IMG_DIR, filename)
    image = cv2.imread(img_path)

    if image is None:
        continue

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    transformed = nighttime_transform(image=image_rgb)
    degraded_image_bgr = cv2.cvtColor(transformed['image'], cv2.COLOR_RGB2BGR)

    save_path = os.path.join(NIGHT_IMG_DIR, f"night_{filename}")
    cv2.imwrite(save_path, degraded_image_bgr)

print(f"\nSuccess! Generated {len(image_files)} nighttime images in /content/nighttime_images")

Generating synthetic nighttime dataset...


Processing images: 100%|██████████| 746/746 [00:07<00:00, 102.05it/s]


Success! Generated 746 nighttime images in /content/nighttime_images


In [5]:
#checking class ID's
import os

LBL_DIR = '/content/source_labels'
class_ids = set()

for lbl_file in os.listdir(LBL_DIR):
    if lbl_file.endswith('.txt'):
        with open(os.path.join(LBL_DIR, lbl_file), 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 1:
                    class_ids.add(int(parts[0]))

print(f"Found class IDs in dataset: {sorted(class_ids)}")
print(f"Max class ID: {max(class_ids) if class_ids else 0}")

Found class IDs in dataset: [0, 1, 2]
Max class ID: 2


In [6]:
import os
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from PIL import Image

IMG_DIR = '/content/source_images'
LBL_DIR = '/content/source_labels'

# 1. Calculate Classes
class_ids = set()
for lbl_file in os.listdir(LBL_DIR):
    if lbl_file.endswith('.txt'):
        with open(os.path.join(LBL_DIR, lbl_file), 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 1:
                    class_ids.add(int(parts[0]))

max_class_id = max(class_ids) if class_ids else 0
NUM_CLASSES = max_class_id + 2  # e.g., max_id 2 -> 2+2 = 4 classes (Background + 0, 1, 2)
print(f"Dataset has YOLO classes: {sorted(class_ids)}")
print(f"Model will use NUM_CLASSES = {NUM_CLASSES} (Background + {len(class_ids)} objects)")

class LicensePlateDataset(Dataset):
    def __init__(self, img_dir, lbl_dir):
        self.img_dir = img_dir
        self.lbl_dir = lbl_dir
        self.images = sorted([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        lbl_path = os.path.join(self.lbl_dir, img_name.rsplit('.', 1)[0] + '.txt')

        image = Image.open(img_path).convert("RGB")
        width, height = image.size

        boxes = []
        labels = []

        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        x_center = float(parts[1])
                        y_center = float(parts[2])
                        w = float(parts[3])
                        h = float(parts[4])

                        x1 = (x_center - w / 2) * width
                        y1 = (y_center - h / 2) * height
                        x2 = (x_center + w / 2) * width
                        y2 = (y_center + h / 2) * height

                        # Clamp boxes
                        x1 = max(0, min(x1, width))
                        y1 = max(0, min(y1, height))
                        x2 = max(0, min(x2, width))
                        y2 = max(0, min(y2, height))

                        if x2 > x1 and y2 > y1:
                            boxes.append([x1, y1, x2, y2])
                            # YOLO class 0 -> PyTorch label 1
                            # YOLO class 2 -> PyTorch label 3
                            labels.append(class_id + 1)

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx]),
            'area': (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
            'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64)
        }

        transform = torchvision.transforms.Compose([
            torchvision.transforms.ToTensor(),
        ])
        image = transform(image)

        return image, target

dataset = LicensePlateDataset(IMG_DIR, LBL_DIR)
data_loader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=0, collate_fn=lambda x: tuple(zip(*x))) # num_workers=0 is safer for debugging

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {device}")

print("Loading model...")
model = fasterrcnn_mobilenet_v3_large_320_fpn(weights="DEFAULT")

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

print("Moving model to device...")
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

num_epochs = 5
print(f"Starting training for {num_epochs} epochs...")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    for images, targets in data_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()

    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss/len(data_loader):.4f}")

print("\nTraining complete!")

Dataset has YOLO classes: [0, 1, 2]
Model will use NUM_CLASSES = 4 (Background + 3 objects)
Using device: cuda
Loading model...
Downloading: "https://download.pytorch.org/models/fasterrcnn_mobilenet_v3_large_320_fpn-907ea3f9.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_mobilenet_v3_large_320_fpn-907ea3f9.pth


100%|██████████| 74.2M/74.2M [00:00<00:00, 88.3MB/s]


Moving model to device...
Starting training for 5 epochs...
Epoch 1/5 | Loss: 0.6449
Epoch 2/5 | Loss: 0.7842
Epoch 3/5 | Loss: 0.7645
Epoch 4/5 | Loss: 0.7730
Epoch 5/5 | Loss: 0.8311

Training complete!
